In [1]:
# One-time deps (skip if already in .venv)
%pip install -q -r "d:/HCMUS_ComputerScience/code/AIC2026/requirements.txt"


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Local Whisper JSONL roots (rglob *.jsonl). Add more Lxx paths as needed.
ASR_DIRS = [
    # r"C:/AIC2026-media/ASR/L21",
    r"C:/AIC2026-media/features/asr/L21",
    r"C:/AIC2026-media/features/asr/L22",
    r"C:/AIC2026-media/features/asr/L23",
    r"C:/AIC2026-media/features/asr/L24",
    r"C:/AIC2026-media/features/asr/L25",
    r"C:/AIC2026-media/features/asr/L26",
    r"C:/AIC2026-media/features/asr/L27",
    r"C:/AIC2026-media/features/asr/L28",
    r"C:/AIC2026-media/features/asr/L29",
    r"C:/AIC2026-media/features/asr/L30",
    
]


In [3]:
# Output: repo features/asr_emb/Lxx/VIDEO_ID.{npy,jsonl} + model.json (L21–L24 working set)
import json
import re
from pathlib import Path

REPO = Path(r"C:/AIC2026-media").resolve()
OUT_ROOT = REPO / "features" / "asr_emb"
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DEVICE = "cpu"  # cuda | cpu
BATCH_SIZE = 8
BATCH_RE = re.compile(r"^(L\d+)_", re.I)

missing = [d for d in ASR_DIRS if not Path(d).is_dir()]
if missing:
    raise SystemExit(f"Not a directory: {missing}")

jsonl_files = []
seen = set()
for d in ASR_DIRS:
    for p in sorted(Path(d).rglob("*.jsonl")):
        key = p.stem
        if key in seen:
            print(f"warning: duplicate stem {key}, keep first", flush=True)
            continue
        seen.add(key)
        jsonl_files.append(p)

if not jsonl_files:
    raise SystemExit(f"No .jsonl under ASR_DIRS={ASR_DIRS}")

print(f"jsonl_count={len(jsonl_files)}")
print("REPO =", REPO)
print("OUT_ROOT =", OUT_ROOT)
print("MODEL_NAME =", MODEL_NAME)
print("DEVICE =", DEVICE)


jsonl_count=873
REPO = C:\AIC2026-media
OUT_ROOT = C:\AIC2026-media\features\asr_emb
MODEL_NAME = sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
DEVICE = cpu


In [4]:
# Segment embed: skip empty text; write filtered jsonl + L2-normalized .npy (row i ↔ line i).
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

out_root = Path(OUT_ROOT)
out_root.mkdir(parents=True, exist_ok=True)

device = DEVICE if (DEVICE == "cpu" or torch.cuda.is_available()) else "cpu"
if DEVICE == "cuda" and device == "cpu":
    print("warning: CUDA requested but unavailable; using cpu", flush=True)

model = SentenceTransformer(MODEL_NAME, device=device)
dim = int(model.get_embedding_dimension())
print(f"model loaded dim={dim} device={device}", flush=True)


def load_segments(path: Path) -> list[dict]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = str(obj.get("text") or "").strip()
        if not text:
            continue
        rows.append(
            {
                "start": float(obj["start"]),
                "end": float(obj["end"]),
                "text": text,
            }
        )
    return rows


def batch_id(stem: str) -> str:
    m = BATCH_RE.match(stem)
    return m.group(1) if m else "unknown"


n_ok = 0
n_empty = 0
for i, src in enumerate(jsonl_files, start=1):
    stem = src.stem
    batch = batch_id(stem)
    dest_dir = out_root / batch
    dest_dir.mkdir(parents=True, exist_ok=True)
    npy_path = dest_dir / f"{stem}.npy"
    jsonl_path = dest_dir / f"{stem}.jsonl"

    segs = load_segments(src)
    if not segs:
        n_empty += 1
        print(f"[{i}/{len(jsonl_files)}] skip empty {stem}", flush=True)
        continue

    texts = [s["text"] for s in segs]
    emb = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32, copy=False)

    if emb.ndim != 2 or emb.shape[0] != len(segs):
        raise SystemExit(f"Bad embed shape {emb.shape} for {stem} n={len(segs)}")

    np.save(npy_path, emb)
    with jsonl_path.open("w", encoding="utf-8") as f:
        for s in segs:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")

    n_ok += 1
    if i == 1 or i % 10 == 0 or i == len(jsonl_files):
        print(
            f"[{i}/{len(jsonl_files)}] {batch}/{stem}  segs={len(segs)}  → {npy_path}",
            flush=True,
        )

meta = {"model": MODEL_NAME, "dim": dim, "normalize": True}
(out_root / "model.json").write_text(
    json.dumps(meta, indent=2) + "\n", encoding="utf-8"
)
print(f"Done: wrote={n_ok}  empty_skipped={n_empty}  dim={dim}")
print("model.json =", out_root / "model.json")


d:\HCMUS_ComputerScience\code\AIC2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5378.11it/s]


model loaded dim=384 device=cpu
[1/873] L21/L21_V001  segs=408  → C:\AIC2026-media\features\asr_emb\L21\L21_V001.npy
[10/873] L21/L21_V011  segs=327  → C:\AIC2026-media\features\asr_emb\L21\L21_V011.npy
[20/873] L21/L21_V022  segs=375  → C:\AIC2026-media\features\asr_emb\L21\L21_V022.npy
[30/873] L22/L22_V001  segs=430  → C:\AIC2026-media\features\asr_emb\L22\L22_V001.npy
[40/873] L22/L22_V011  segs=501  → C:\AIC2026-media\features\asr_emb\L22\L22_V011.npy
[50/873] L22/L22_V021  segs=293  → C:\AIC2026-media\features\asr_emb\L22\L22_V021.npy
[60/873] L22/L22_V031  segs=244  → C:\AIC2026-media\features\asr_emb\L22\L22_V031.npy
[70/873] L23/L23_V010  segs=103  → C:\AIC2026-media\features\asr_emb\L23\L23_V010.npy
[80/873] L23/L23_V020  segs=53  → C:\AIC2026-media\features\asr_emb\L23\L23_V020.npy
[90/873] L24/L24_V006  segs=1  → C:\AIC2026-media\features\asr_emb\L24\L24_V006.npy
[92/873] skip empty L24_V008
[97/873] skip empty L24_V013
[99/873] skip empty L24_V015
[100/873] skip empty L24_